# 04b - production pipeline v1: model outputs + OCR + post-rules R1/R2/R3 (thin wrapper)

v1 = the 04 safety submission (v0: Qwen3.5-4B, image only, KEYED output, prompt v2, greedy, seed
42, fp16, logprobs on) + the OCR text of the 280 test pages (PaddleOCR PP-OCRv5 server, own venv)
+ post-processing rules R1 (carrier from the model's own supplier slot), R2 (waybill pattern
backfill from OCR), R3 (invoice slot-shape swap with the frozen `meta/slot_shapes.json`), all
three ON (`reports/v1_replay.md`, `reports/rule_gate.md`). Test images only: there are no test
labels, nothing here scores anything, and no document value is printed (counts, ids, hashes).

Flow: setup (Drive, clone at the pin, unzip data incl. test images, 200 docs / 280 pages by file
name) -> ESTIMATE (T4 hours / CU for the VLM part if it must run, and the OCR part) -> base env
(`uv sync --frozen`, NO vlm group, NO paddle) -> plan -> OCR stage (own venv at /content/ocr-venv
via a subprocess: GPU with a loud CPU fallback, resumable per page, 280/280 asserted,
`ocr_timing.json`, a 5-page seeded re-OCR in a fresh process) -> REUSE DECISION -> either
(a) REUSE the v0 model outputs, or (b) FULL INFERENCE (VLM env, the same smoke gate, batch size
contract, resumable run and determinism pass as 04) -> assemble with R1-R3 -> validation
(the blocking checks of 04 + the v1 checks) -> banner VALIDATED / REJECTED.

REUSE is allowed ONLY when a pure function (`shipdoc.reuse.decide_reuse`) finds all of: the v0
folder (`REUSE_V0_DIR`) holds a manifest whose validation report says ok; v0 config hash,
prompt hash, model revision, seed, output format, arm, shard and BATCH SIZE equal v1's; the v0
trace has exactly the 200 test ids with complete pages and logprobs; and the DECODE-PATH
FINGERPRINT holds. The v0 code SHA legitimately differs from this pin, so code identity is
REPLACED by: the git blob hash of every file on the model-output path is identical at the v0 SHA
and at HEAD, `spike.py` differs only by the allowlisted opt-in additions (`header_hint`,
`rule_cfg`; default off), plus the config and prompt hashes. Anything else REFUSES, loudly, and
the notebook runs the full inference path instead; reuse is never silent. In reuse mode the VLM is
not run: its determinism rests on v0's own check (re-attached to the manifest with its hash).

Parameters: `PINNED_SHA`; `BATCH_SIZE` (8 = the 02/04 production size; must equal v0's for reuse
and the dev run's); `SHARD` / `MODE` ("run" or "merge", full inference only; reuse needs "0/1");
`REUSE_V0_DIR` (default `submissions/v0_42b812b` under MyDrive/shipdoc-extract, None = always
full inference); `REUSE_ACK_SPIKE_DIFF` (False: a `spike.py` diff that is not the allowlisted
one refuses reuse; True accepts any such diff, recorded in the manifest); `DEV_RUN_DIR` as 04.

Output: `MyDrive/shipdoc-extract/submissions/v1_<sha7>/` with `test_predictions.json`,
`trace.jsonl`, `manifest.json`, `validation_report.json`, `rules.jsonl`, `ocr_timing.json`; on a
failed check the predictions are `test_predictions.REJECTED.json`. Download by hand into
`$SHIPDOC_SUBMISSIONS_DIR\v1_<sha7>\`. If the session dies, Run all: every stage resumes.
See notebooks/README.md.


In [ ]:
# Parameters. PINNED_SHA = the repo commit this run is pinned to (full 40-char SHA).
PINNED_SHA = "8833c73e799dcce4372dc72f133835072a0b8126"
RUN_NAME = "test"
CONFIG = "qwen35_4b_img_only"  # KEYED output format, prompt v2, image only: the production pick
SPLIT = "test"
EXPECTED_DOCS = 200
EXPECTED_PAGES = 280
SMOKE_DOCS = "splits/smoke5.json"  # 5 dev docs picked by meta tags (shipdoc.smoke)
BENCH_DOCS = "splits/bench12.json"  # 12 dev pages for the batch-size bench (shipdoc.bench)
BATCH_SIZE = 8  # the 02/04 production batch size; must equal v0's (reuse) and the dev run's
SHARD = "0/1"  # "i/K": full inference only; reuse needs "0/1"
MODE = "run"  # "run" = OCR, reuse decision, (inference), assemble; "merge" = combine K shards
REUSE_V0_DIR = "submissions/v0_42b812b"  # under MyDrive/shipdoc-extract, or None = no reuse
REUSE_ACK_SPIKE_DIFF = False  # True accepts a spike.py diff beyond the allowlisted opt-in ones
DEV_RUN_DIR = "zeroshot500_qwen35_4b_img_only_keyed_42b812b"  # dev run folder under runs/, or None

if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell (the pin commit fills it).")
if MODE not in ("run", "merge"):
    raise ValueError(f"MODE must be 'run' or 'merge', got {MODE!r}")
if BATCH_SIZE is not None and (isinstance(BATCH_SIZE, bool) or not isinstance(BATCH_SIZE, int)
                               or BATCH_SIZE < 1):
    raise ValueError(f"BATCH_SIZE must be None or an int >= 1, got {BATCH_SIZE!r}")
if REUSE_V0_DIR is not None and not isinstance(REUSE_V0_DIR, str):
    raise ValueError(f"REUSE_V0_DIR must be None or a path string, got {REUSE_V0_DIR!r}")
if not isinstance(REUSE_ACK_SPIKE_DIFF, bool):
    raise ValueError(f"REUSE_ACK_SPIKE_DIFF must be True or False, got {REUSE_ACK_SPIKE_DIFF!r}")
_i, _, _k = SHARD.partition("/")
if not (_i.isdigit() and _k.isdigit() and int(_k) >= 1 and int(_i) < int(_k)):
    raise ValueError(f"SHARD must look like 'i/K' with 0 <= i < K, got {SHARD!r}")
SHARD_I, SHARD_K = int(_i), int(_k)
if MODE == "merge" and SHARD_K < 2:
    raise ValueError("MODE='merge' combines K shard folders: set SHARD to '0/K' with K >= 2.")
SHA7 = PINNED_SHA[:7]
RUN_ID = f"{RUN_NAME}_{CONFIG}_keyed_{SHA7}"  # the unsharded run (and the merge target)
SHARD_RUN_ID = RUN_ID if SHARD_K == 1 else f"{RUN_ID}_shard{SHARD_I}of{SHARD_K}"  # this tab
TAB = f"{SHARD_I}of{SHARD_K}"  # suffix of the files that belong to this tab only


## Account

No account check in this public build: Drive is mounted as whichever Google account you sign in
with, and the inputs must be in that account's `MyDrive/shipdoc-extract/` (see the repository
README, "Reproduce from images").


In [ ]:
import subprocess
from pathlib import Path

from google.colab import drive

DRIVE_DIR = Path("/content/drive/MyDrive/shipdoc-extract")
REQUIRED_ZIPS = ["data.zip", "assignment.zip"]
RUNS_DIR = DRIVE_DIR / "runs"

drive.mount("/content/drive")

print("Public build: any Google account; inputs are read from", DRIVE_DIR)

missing = [n for n in REQUIRED_ZIPS if not (DRIVE_DIR / n).is_file()]
if missing:
    raise FileNotFoundError(f"Missing in MyDrive/shipdoc-extract/: {', '.join(missing)}")
print("Drive inputs present:", ", ".join(REQUIRED_ZIPS))
RUNS_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSIONS_DIR = DRIVE_DIR / "submissions"  # test predictions live here, not in the repo
SUBMISSION_DIR = SUBMISSIONS_DIR / f"v1_{SHA7}"
META_DIR = RUNS_DIR / f"{RUN_ID}_meta"  # plan, smoke status and batch decision of the tabs
SUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)
META_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = RUNS_DIR / (RUN_ID if MODE == "merge" else SHARD_RUN_ID)
PLAN_PATH = META_DIR / "plan.json"
TEST_DOCS_PATH = META_DIR / "test_docs.json"  # ids only: the doc list of merge-shards
SMOKE_STATUS_PATH = META_DIR / f"smoke_status_{TAB}.json"
DECISION_PATH = META_DIR / f"batch_decision_{TAB}.json"
# The dev run whose manifest holds the batch size the test run must match (None = no comparison).
DEV_RUN = None
if DEV_RUN_DIR:
    DEV_RUN = Path(DEV_RUN_DIR) if Path(DEV_RUN_DIR).is_absolute() else RUNS_DIR / DEV_RUN_DIR
    if not (DEV_RUN / "manifest.json").is_file():
        print("!" * 78)
        print(f"DEV RUN NOT FOUND: {DEV_RUN} has no manifest.json. The bench is used as is and the")
        print("batch-size contract with the dev run is NOT checked (the manifest records this).")
        print("!" * 78)
        DEV_RUN = None
else:
    print("DEV_RUN_DIR is None: no dev run to compare the batch size with (recorded as unchecked).")
print("run_id:", RUN_ID, "| mode:", MODE, "| shard:", SHARD, "| batch size:", BATCH_SIZE or "bench")
print("Drive run folder:", RUN_DIR)
print("Drive submission folder:", SUBMISSION_DIR)
OCR_ROOT = DRIVE_DIR / "ocr_cache_test"  # <root>/paddleocr/test/<page>.json, resumable
OCR_ROOT.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
V0_DIR = None  # the 04 submission folder whose model outputs may be reused (decided below)
if REUSE_V0_DIR:
    V0_DIR = Path(REUSE_V0_DIR) if Path(REUSE_V0_DIR).is_absolute() else DRIVE_DIR / REUSE_V0_DIR
    print("v0 folder (reuse candidate):", V0_DIR, "| manifest present:",
          (V0_DIR / "manifest.json").is_file())
print("OCR cache folder (Drive):", OCR_ROOT)


In [ ]:
import os

from google.colab import userdata


def get_secret(name: str, required: bool) -> str | None:
    """Read a Colab secret without ever printing it."""
    try:
        value = userdata.get(name)
    except Exception:  # SecretNotFoundError / NotebookAccessError: both mean "not usable"
        value = None
    if required and not value:
        raise RuntimeError(
            f"Colab secret {name} is missing or notebook access is off "
            "(left sidebar -> key icon -> add it and enable Notebook access)."
        )
    return value or None


HF_TOKEN = get_secret("HF_TOKEN", required=False)
print("HF_TOKEN:", "set" if HF_TOKEN else "not set (optional)")


In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/gaurav-gandhi-2411/shipdoc-extract-public.git"
REPO = Path("/content/shipdoc-extract")
if not PINNED_SHA or "FILL" in PINNED_SHA:
    raise ValueError("Set PINNED_SHA in the parameters cell to the commit this run is pinned to.")
if "FILL" in REPO_URL:
    raise ValueError("REPO_URL is a placeholder: the public repository is not published yet.")


def git(*args: str) -> str:
    """Run git in the repo dir (or /content). The repository is public: no credentials at all."""
    cwd = REPO if (REPO / ".git").is_dir() else "/content"
    res = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, check=False)
    if res.returncode:
        raise RuntimeError(f"git {args[0]} failed (exit {res.returncode}): {res.stderr[-800:]}")
    return res.stdout.strip()


if (REPO / ".git").is_dir():
    git("fetch", "origin")
else:
    git("clone", REPO_URL, str(REPO))
git("checkout", PINNED_SHA)
head = git("rev-parse", "HEAD")
pinned_full = git("rev-parse", f"{PINNED_SHA}^{{commit}}")
assert head == pinned_full, f"HEAD {head} != PINNED_SHA {pinned_full}"
print("HEAD =", head)


In [ ]:
import json
import re
import shutil
import zipfile

LOCAL_ZIPS = Path("/content/zips")
LOCAL_ZIPS.mkdir(exist_ok=True)
CONTENT = Path("/content")
# data.zip holds data/ and assignment.zip holds assignment/; both unzip to /content (not into the
# clone), and the CLI is pointed at them with SHIPDOC_DATA_DIR / SHIPDOC_ASSIGNMENT_DIR below.
for name in ("data.zip", "assignment.zip"):
    local = LOCAL_ZIPS / name
    if not local.is_file() or local.stat().st_size != (DRIVE_DIR / name).stat().st_size:
        shutil.copyfile(DRIVE_DIR / name, local)  # local disk: Drive reads are slow
    with zipfile.ZipFile(local) as zf:
        files = [n for n in zf.namelist() if not n.endswith("/")]
        if any(not (CONTENT / n).is_file() for n in files):  # also repairs a half-extracted tree
            zf.extractall(CONTENT)
    print(f"{name}: {len(files)} files under {CONTENT / files[0].split('/')[0]}")

DATA_DIR, ASSIGNMENT_DIR = CONTENT / "data", CONTENT / "assignment"
DEV_LABELS = DATA_DIR / "dev" / "labels"  # the smoke gate and the bench use DEV docs only
TEST_IMAGES = DATA_DIR / SPLIT / "images"
SCHEMA_PATH = ASSIGNMENT_DIR / "schema.json"
SAMPLE_PATH = ASSIGNMENT_DIR / "sample_submission.json"
assert SCHEMA_PATH.is_file(), "assignment.zip has no schema.json"
assert TEST_IMAGES.is_dir(), f"data.zip has no {TEST_IMAGES}: the test images are required"
# There are no test labels and this notebook never looks for any.
assert not (DATA_DIR / SPLIT / "labels").exists(), "unexpected test labels in data.zip: stop"
STEM = re.compile(r"^(.+)_p(\d+)$")
_pages = {}
for _p in TEST_IMAGES.iterdir():
    _m = STEM.match(_p.stem)
    if _p.is_file() and _m:
        _pages[_m.group(1)] = _pages.get(_m.group(1), 0) + 1  # ids from FILE NAMES only
test_pages_by_doc = dict(sorted(_pages.items()))
assert len(test_pages_by_doc) == EXPECTED_DOCS, f"{len(test_pages_by_doc)} test docs found"
assert sum(test_pages_by_doc.values()) == EXPECTED_PAGES, "test page count differs"
print(f"data OK: {EXPECTED_DOCS} test docs / {EXPECTED_PAGES} page images (no labels); dev labels:",
      DEV_LABELS.is_dir())


# Public build: the evaluators' package (assignment.zip, never vendored in the repository) must be
# complete BEFORE the install and model cells; the post-processing rules read its schema.json.
for _need in ("schema.json", "score.py"):
    assert (ASSIGNMENT_DIR / _need).is_file(), (
        f"assignment.zip has no assignment/{_need}: upload BOTH data.zip and assignment.zip "
        "(your evaluator package) to MyDrive/shipdoc-extract/"
    )


In [ ]:
if MODE == "run":
    # T4-hour and compute-unit (CU) estimate BEFORE anything runs on the GPU: read it.
    # No confirmation prompt (Run all must not block): interrupt the run yourself if it is too high.
    import importlib.util
    import json
    import subprocess

    _ge_path = REPO / "scripts" / "gpu_estimate.py"
    _spec = importlib.util.spec_from_file_location("gpu_estimate", _ge_path)
    ge = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(ge)

    SPEED = json.loads((REPO / "configs" / "spike_speed.json").read_text(encoding="utf-8"))
    smoke_ids = json.loads((REPO / SMOKE_DOCS).read_text(encoding="utf-8"))


    def read_gpu_name():
        try:
            out = subprocess.run(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                capture_output=True, text=True, timeout=30, check=False,
            ).stdout.strip().splitlines()
        except (OSError, subprocess.TimeoutExpired):
            return None
        return out[0].strip() if out else None


    def test_estimate_table(speed, config, pages_by_doc, smoke_pages, gpu_name):
        """ESTIMATE (UNVERIFIED) hours and CU of the v0 test run per batch size, 1 and 2 tabs."""
        m = speed["models"][ge.base_config(config)]
        n_out = float(m["n_output_tokens_mean"])
        load_s = float(speed.get("model_load_s", 0))
        r1, r2 = speed.get("t4_cu_per_hour"), speed.get("t4_cu_per_hour_conservative")
        n_pages, n_docs = sum(pages_by_doc.values()), len(pages_by_doc)
        multi = [n for n in pages_by_doc.values() if n > 1]
        # 2 multi-page + 3 single-page documents are re-decoded (shipdoc.predict.DET_RULE)
        det_pages = round(2 * sum(multi) / len(multi) + 3) if multi else 5

        def cu(rate, hours):
            return "n/a" if rate is None else f"{hours * rate:.1f}"

        bar = "=" * 124
        head = (f"ESTIMATE (UNVERIFIED) T4 hours / CU: v1 VLM run, {n_docs} docs / {n_pages} "
                f"pages, {config} (keyed). Batching gain ASSUMED until the bench measures it")
        lines = [bar, head, bar]
        warn = ge.gpu_warning(gpu_name)
        if warn:
            lines.append(warn)
        lines += [
            f"per tab: model load {load_s:.0f} s (ESTIMATE) for the smoke ({smoke_pages} pages, "
            f"batch 1), the test run and the determinism pass ({det_pages} pages, a fresh "
            "process = one more load); 'bench reused' = the dev bench result applies (no bench), "
            "'bench rerun' = worst case (one load + 12 dev pages at batch 1/2/4/8).",
            f"s/page = prefill_s {m['prefill_s']} + {n_out} tok x straggler(B) / "
            f"({m['decode_tok_s']} tok/s x eff x B), straggler = {ge.STRAGGLER}; eff = scaling "
            "efficiency of the decode.",
            "",
            f"{'scenario':<22} {'K':>1} {'s/page':>6} | {'wall h':>6} {'CU@' + str(r1):>8} "
            f"{'CU@' + str(r2):>8} | {'wall h':>6} {'CU@' + str(r1):>8} {'CU@' + str(r2):>8}",
            f"{'':<22} {'':>1} {'':>6} | {'bench reused':^24} | {'bench rerun':^24}",
        ]
        for b, eff in ge.batch_scenarios():
            name = "B=1 (no bench)" if b == 1 else f"B={b} @ {int(eff * 100)}% scaling"
            s = ge.batch_s_per_page(m["prefill_s"], m["decode_tok_s"], n_out, b, eff)
            for k in ge.SHARD_COUNTS:
                det_h = (load_s + det_pages * s) / 3600
                cells = []
                for rerun in (False, True):
                    h = ge.tab_hours(speed, config, n_out, b, eff, n_pages / k,
                                     smoke_pages=smoke_pages, with_bench=rerun and b > 1) + det_h
                    cells.append(f"{h:6.2f} {cu(r1, k * h):>8} {cu(r2, k * h):>8}")
                lines.append(f"{name:<22} {k:>1} {s:6.1f} | {cells[0]} | {cells[1]}")
        lines += [
            "",
            "wall h = one tab; CU = K tabs summed (K = 2: two Colab tabs, MODE='merge' after).",
            f"CU rate {r1}/h: {speed.get('t4_cu_source', 'source unset')}",
            f"CU rate {r2}/h: {speed.get('t4_cu_conservative_source', 'source unset')}",
            "Not modelled: logprob capture overhead (assumed 0), Drive I/O, session restarts, OOM "
            "fallbacks, the validation step (CPU, seconds).",
            bar,
        ]
        return "\n".join(lines)


    GPU_NAME = read_gpu_name()
    print("Runtime GPU (nvidia-smi):", GPU_NAME)
    smoke_pages = ge.count_pages(smoke_ids, DEV_LABELS)
    print(test_estimate_table(SPEED, CONFIG, test_pages_by_doc, smoke_pages, GPU_NAME))
    print(f"THIS TAB: shard {SHARD}, batch size {BATCH_SIZE or 'from the bench'}, mode {MODE}.")

else:
    print("MODE = merge: skipped (estimate)")


In [ ]:
# ESTIMATE (UNVERIFIED) of the OCR stage: 280 test pages, PaddleOCR PP-OCRv5
# server
# with the document + text-line orientation classifiers (shipdoc.ocr). No GPU measurement of
# it exists in this repo: the GPU figure is an ASSUMPTION. The CPU figure is the only
# measurement: 10.16 s/page mean over 951 pages (reports/ocr.md, local 8-thread CPU, shared
# with other jobs); a free Colab
# runtime has fewer cores, so the Colab CPU figure below doubles it (also an ASSUMPTION).
import json

OCR_GPU_S_PER_PAGE = 2.0  # ASSUMED, not measured
OCR_CPU_S_PER_PAGE = 2 * 10.16  # local measurement x 2 (ASSUMED Colab CPU slowdown)
OCR_SETUP_MIN = 10.0  # ASSUMED: venv build + paddle wheel + model download + first model load
_speed = json.loads((REPO / "configs" / "spike_speed.json").read_text(encoding="utf-8"))
_rate = _speed.get("t4_cu_per_hour")
_gpu_h = (OCR_SETUP_MIN * 60 + EXPECTED_PAGES * OCR_GPU_S_PER_PAGE) / 3600
_cpu_h = (OCR_SETUP_MIN * 60 + EXPECTED_PAGES * OCR_CPU_S_PER_PAGE) / 3600
print("=" * 100)
print("ESTIMATE (UNVERIFIED) OCR stage, resumable per page (pages already cached are skipped):")
print(f"  GPU (T4): {_gpu_h:.2f} h incl. {OCR_SETUP_MIN:.0f} min setup "
      f"({OCR_GPU_S_PER_PAGE} s/page ASSUMED), ~{_gpu_h * _rate:.2f} CU @ {_rate} CU/h")
print(f"  CPU fallback: {_cpu_h:.2f} h ({OCR_CPU_S_PER_PAGE:.1f} s/page ASSUMED), CU rate of a "
      "CPU runtime is not in configs/spike_speed.json: n/a")
print("  + a seeded 5-page re-OCR in a fresh process (about one model load + 5 pages).")
print(f"REUSE_V0_DIR = {REUSE_V0_DIR!r}: if the reuse decision allows it the VLM part costs 0 GPU "
      "hours (assembly is CPU, seconds); else see the VLM table above.")
print("=" * 100)


In [ ]:
import collections
import subprocess

PY = str(REPO / ".venv" / "bin" / "python")
ENV = {
    **os.environ,
    "SHIPDOC_PROFILE": "colab",
    "HF_HOME": "/content/hf_cache",
    "SHIPDOC_RUNS_DIR": str(RUNS_DIR),
    "SHIPDOC_DATA_DIR": str(DATA_DIR),
    "SHIPDOC_ASSIGNMENT_DIR": str(ASSIGNMENT_DIR),
    "SHIPDOC_OCR_CACHE": str(OCR_ROOT),  # the test OCR cache that R2 reads
    "SHIPDOC_SUBMISSIONS_DIR": str(SUBMISSIONS_DIR),
    "SHIPDOC_TMP_DIR": "/content/tmp",
    "UV_CACHE_DIR": "/content/uv_cache",
}
if HF_TOKEN:
    ENV["HF_TOKEN"] = HF_TOKEN
Path(ENV["SHIPDOC_TMP_DIR"]).mkdir(exist_ok=True)


def run_stream(
    cmd: list[str], tail: int = 40, env: dict | None = None
) -> tuple[int, list[str]]:
    """Run a command, echo its output live, return (exit code, last `tail` lines)."""
    last: collections.deque[str] = collections.deque(maxlen=tail)
    with subprocess.Popen(
        cmd, cwd=REPO, env=env or ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as proc:
        for line in proc.stdout:
            print(line, end="")
            last.append(line.rstrip("\n"))
    return proc.returncode, list(last)


if subprocess.run(["uv", "--version"], capture_output=True, check=False).returncode:
    subprocess.run(["pip", "install", "-q", "uv"], check=True)
rc, _ = run_stream(["uv", "sync", "--frozen", "--python", "3.11"])
assert rc == 0, f"uv sync failed (exit {rc})"
# Paddle must never be in this environment (it is its own dependency group, installed only into
# the separate OCR venv by scripts/colab_ocr_test.sh).
_no_ocr = (
    "import importlib.util as u, sys; sys.exit(0 if u.find_spec('paddle') is None else 1)"
)
assert subprocess.run([PY, "-c", _no_ocr], env=ENV, check=False).returncode == 0, (
    "the OCR stack is in the decoding venv: Paddle runs ONLY in /content/ocr-venv"
)


In [ ]:
# Plan: document ids come from the test image FOLDER NAMES (never labels); the shard's
# documents and the 5 seeded determinism documents are fixed here (rule in the output).
rc, tail = run_stream([PY, "-m", "shipdoc", "predict", "plan", "--shard", SHARD,
                       "--out", str(PLAN_PATH)])
assert rc == 0, f"plan failed (exit {rc}): {tail[-3:]}"
plan = json.loads(PLAN_PATH.read_text())
assert plan["n_docs"] == EXPECTED_DOCS and plan["n_pages"] == EXPECTED_PAGES, "plan counts differ"
assert sorted(plan["docs"]) == sorted(test_pages_by_doc), "plan ids differ from the file names"
TEST_DOCS_PATH.write_text(json.dumps(plan["docs"]))
print("determinism rule:", plan["determinism_rule"])


In [ ]:
# OCR of the 280 TEST pages, in its OWN venv (paddlepaddle-gpu clashes with the vlm
# group, so Paddle is never installed into or imported by the environment that decodes: this
# cell only starts subprocesses). Resumable: pages already cached AND valid are skipped (an
# invalid page file is renamed to *.invalid, never deleted, and redone). GPU when the runtime
# has one and the GPU wheel installed, else CPU with a LOUD warning. OCR text is never
# printed: counts and seconds only.
# Asserts 280/280 valid cached pages, writes ocr_timing.json (pages, seconds total, seconds/page
# mean and p95, device) to the submission folder, then re-OCRs 5 seeded pages in a fresh process.
import json
import subprocess
import time

OCR_VENV = Path("/content/ocr-venv")  # never the venv that decodes
assert OCR_VENV.resolve() != (REPO / ".venv").resolve(), "OCR must not share the decoding venv"
OCR_SH = ["bash", str(REPO / "scripts" / "colab_ocr_test.sh")]
OCR_ENV = {**ENV, "OCR_VENV": str(OCR_VENV)}
OCR_TIMING_PATH = SUBMISSION_DIR / "ocr_timing.json"
OCR_RECHECK_PATH = META_DIR / "ocr_recheck.json"
STAGE = [PY, "-m", "shipdoc.ocr_stage"]
PAGE_ARGS = ["--images-dir", str(TEST_IMAGES), "--cache-root", str(OCR_ROOT),
             "--expect-pages", str(EXPECTED_PAGES)]


def has_gpu() -> bool:
    try:
        res = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, timeout=30,
                             check=False)
    except (OSError, subprocess.TimeoutExpired):
        return False
    return res.returncode == 0 and "GPU" in res.stdout


def ocr_device() -> dict:
    """Device from the runtime and the OCR venv's paddle flavour; loud on a CPU fallback."""
    flavor_file = OCR_VENV / ".paddle_flavor"
    flavor = flavor_file.read_text().strip() if flavor_file.is_file() else ""
    res = subprocess.run([*STAGE, "device", "--has-gpu", "1" if has_gpu() else "0",
                          "--flavor", flavor], cwd=REPO, env=ENV, capture_output=True, text=True,
                         check=False)
    info = json.loads(res.stdout.strip().splitlines()[-1])
    for warning in info["warnings"]:
        print("!" * 78)
        print("OCR DEVICE WARNING:", warning)
        print("!" * 78)
    if not info["ok"]:
        raise RuntimeError(f"OCR cannot run on this runtime: {info['warnings']}")
    print(f"OCR device: {info['device']} (paddle flavour {info['flavor'] or 'unknown'})")
    return info


def ocr_status(tag: str) -> dict:
    path = META_DIR / f"ocr_check_{tag}.json"
    rc, _ = run_stream([*STAGE, "check", *PAGE_ARGS, "--isolate-invalid", "--out", str(path)])
    return {"rc": rc, **json.loads(path.read_text())}


before = ocr_status("before")
wall, device = None, None
if before["rc"] == 0:
    print(f"OCR cache already complete ({before['valid']}/{EXPECTED_PAGES}): no OCR run")
else:
    print(f"OCR cache: {before['valid']}/{EXPECTED_PAGES} valid; running the missing pages")
    t_ocr = time.time()
    rc, tail = run_stream([*OCR_SH, "setup"], env=OCR_ENV)
    assert rc == 0, f"OCR venv setup failed (exit {rc}): {tail[-3:]}"
    device = ocr_device()["device"]
    rc, tail = run_stream([*OCR_SH, "run", str(DATA_DIR), str(OCR_ROOT)], env=OCR_ENV)
    wall = time.time() - t_ocr
    print(f"--- OCR run exited {rc} after {wall / 60:.1f} min (the check below decides) ---")
after = ocr_status("after")
if after["rc"] != 0:
    raise RuntimeError(
        f"OCR INCOMPLETE: {after['valid']}/{EXPECTED_PAGES} valid pages, {after['missing']} "
        f"missing {after['missing_pages']}, {after['invalid']} invalid. Everything cached is on "
        "Drive: Run all again resumes the rest."
    )
assert after["valid"] == EXPECTED_PAGES, f"{after['valid']} cached OCR pages, not {EXPECTED_PAGES}"
prior = json.loads(OCR_TIMING_PATH.read_text()) if OCR_TIMING_PATH.is_file() else {}
device = device or prior.get("device") or "unknown (the cache was complete at the start)"
fin = [*STAGE, "finalize", *PAGE_ARGS, "--device", device, "--out", str(OCR_TIMING_PATH)]
if wall is not None:
    fin += ["--wall-s", f"{wall:.1f}", "--new-pages", str(after["valid"] - before["valid"])]
rc, tail = run_stream(fin)
assert rc == 0, f"ocr timing failed (exit {rc}): {tail[-3:]}"
assert subprocess.run([PY, "-c", "import importlib.util as u, sys; "
                       "sys.exit(0 if u.find_spec('paddle') is None else 1)"],
                      env=ENV, check=False).returncode == 0, "Paddle leaked into the decoding venv"

# OCR determinism: 5 seeded pages (random.Random(42)) OCR'd AGAIN in a fresh process (the OCR
# venv, into a scratch cache); blocking criterion = identical page text (what R2 reads), the whole
# page JSON minus `seconds` is reported too. A failure is recorded and REJECTS the submission.
recheck = json.loads(OCR_RECHECK_PATH.read_text()) if OCR_RECHECK_PATH.is_file() else {}
if recheck.get("ok") is True:
    print("OCR recheck already passed:", recheck["n_text_identical"], "/", recheck["n_pages"])
else:
    rc, tail = run_stream([*OCR_SH, "setup"], env=OCR_ENV)  # a no-op unless the VM is new
    assert rc == 0, f"OCR venv setup failed (exit {rc}): {tail[-3:]}"
    ocr_device()
    rc, tail = run_stream([*OCR_SH, "module", "recheck", *PAGE_ARGS, "--scratch-root",
                           "/content/ocr_recheck", "--out", str(OCR_RECHECK_PATH)], env=OCR_ENV)
    if rc != 0:
        print("!" * 78)
        print("OCR RE-CHECK DID NOT PASS (exit", rc, "): the submission will be REJECTED. Report")
        print("back with", OCR_RECHECK_PATH, "(page names and booleans only).")
        print("!" * 78)


In [ ]:
# REUSE DECISION (CPU, pure function shipdoc.reuse.decide_reuse). The v0 model outputs
# are reused ONLY when every condition holds; one refusal falls back to FULL INFERENCE,
# loudly (never silent). Code identity cannot hold literally (the v0 pin is older than this
# pin): it is replaced by the decode-path fingerprint = git blob hashes of every file on the
# model-output path identical at the v0 SHA and HEAD (spike.py: only the allowlisted opt-in
# additions) + config / prompt hashes.
# `git diff --stat <v0 sha> <pin> -- src configs` is printed below.
REUSE_OK = False
if V0_DIR is None:
    print("REUSE_V0_DIR is None: full inference.")
elif MODE != "run" or SHARD_K != 1:
    print(f"MODE {MODE!r} / SHARD {SHARD!r}: reuse needs MODE 'run' and SHARD '0/1': full "
          "inference.")
elif BATCH_SIZE is None:
    print("BATCH_SIZE is None (the bench decides): reuse needs an explicit size: full inference.")
else:
    reuse_cmd = [PY, "-m", "shipdoc.reuse", "check", "--v0-dir", str(V0_DIR),
                 "--batch-size", str(BATCH_SIZE), "--shard", SHARD,
                 "--out", str(META_DIR / "reuse_decision.json")]
    if REUSE_ACK_SPIKE_DIFF:
        reuse_cmd.append("--ack-spike-diff")
        print("REUSE_ACK_SPIKE_DIFF = True: a spike.py diff beyond the allowlist is accepted.")
    rc, tail = run_stream(reuse_cmd)
    REUSE_OK = rc == 0
bar = "=" * 78
if REUSE_OK:
    print(bar)
    print("REUSE ALLOWED: the 04 model outputs are reused; the VLM is NOT run (no GPU needed).")
    print("Code identity is REPLACED by the decode-path blob-hash fingerprint + prompt/config "
          "hashes.")
    print("VLM determinism rests on v0's own determinism check (re-attached to the manifest).")
    print(bar)
else:
    print("!" * 78)
    print("REUSE NOT USED -> FULL INFERENCE on the GPU (smoke gate, batch contract, run, "
          "determinism).")
    if V0_DIR is not None and (META_DIR / "reuse_decision.json").is_file():
        print("refusals:", json.loads((META_DIR / "reuse_decision.json").read_text())["reasons"])
    print("!" * 78)
NEED_INFERENCE = MODE == "run" and not REUSE_OK


In [ ]:
if NEED_INFERENCE:
    rc, _ = run_stream(["uv", "sync", "--frozen", "--group", "vlm", "--python", "3.11"])
    assert rc == 0, f"uv sync (vlm) failed (exit {rc})"
    if MODE == "run":
        subprocess.run(["nvidia-smi"], check=False)
        PROBE = (
            "import json, importlib.metadata as m, torch;"
            "print(json.dumps({'torch': torch.__version__, 'cuda': torch.cuda.is_available(),"
            " 'arch': torch.cuda.get_arch_list(),"
            " **{p: m.version(p) for p in ('transformers', 'xgrammar', 'pycountry')}}))"
        )
        info = json.loads(subprocess.run([PY, "-c", PROBE], env=ENV, capture_output=True, text=True,
                                         check=True).stdout.strip().splitlines()[-1])
        print(info)
        assert info["cuda"], "No CUDA device: Runtime -> Change runtime type -> T4 GPU."
        # T4 is sm_75: need a native sm_75 kernel or PTX (compute_XX with XX <= 75) it can JIT.
        ok = "sm_75" in info["arch"] or any(
            a.startswith("compute_") and int(a.split("_")[1]) <= 75 for a in info["arch"]
        )
        assert ok, f"torch build has no sm_75 support: {info['arch']}"

    _no_ocr = (
        "import importlib.util as u, sys; sys.exit(0 if u.find_spec('paddle') is None else 1)"
    )
    assert subprocess.run([PY, "-c", _no_ocr], env=ENV, check=False).returncode == 0, (
        "the OCR stack is in the decoding venv: Paddle runs ONLY in /content/ocr-venv"
    )

else:
    print("skipped (VLM environment): NEED_INFERENCE is false")


In [ ]:
if NEED_INFERENCE:
    # MANDATORY smoke gate (same run and same 7 checks as notebook 02: keyed format, prompt
    # v2, logprobs, 5 DEV docs): `shipdoc predict smoke` writes a status file bound to this
    # code, config and model revision; the test run refuses to start unless it says `passed`.
    # A failure raises.
    SMOKE_RUN_ID = f"smoke_{SHARD_RUN_ID}"
    CFG = f"configs/spike_{CONFIG}.yaml"
    smoke_cmd = [PY, "-m", "shipdoc", "predict", "smoke", "--config", CFG,
                 "--docs", SMOKE_DOCS, "--run-id", SMOKE_RUN_ID, "--status", str(SMOKE_STATUS_PATH)]
    rc, tail = run_stream(smoke_cmd)
    SMOKE_PASSED = rc == 0
    if not SMOKE_PASSED:
        raise RuntimeError(
            "SMOKE GATE FAILED: the test run is NOT started. GG: report back with "
            f"{SMOKE_STATUS_PATH} and the per-assertion table printed above (dev documents only). "
            "Do not edit thresholds to make it pass."
        )
    print("SMOKE GATE PASSED: continuing.")

else:
    print("skipped (smoke gate): NEED_INFERENCE is false")


In [ ]:
if NEED_INFERENCE:
    # BATCH SIZE. BATCH_SIZE = None: `shipdoc predict batch` reuses the bench result when it
    # applies to this code, config and model (the dev run's stored result is copied and verified
    # first), else runs the bench on the 12 dev pages exactly as notebook 02 (largest size with 100%
    # byte-identical outputs vs batch 1 and peak VRAM <= 14.5 GiB, ...). Either way the size must
    # EQUAL the dev run's (shipdoc.runmeta): greedy outputs are only comparable at the same batch
    # size. A mismatch raises.
    if not SMOKE_PASSED:
        raise RuntimeError("The smoke gate has not passed: batch size and test run are NOT run.")
    BENCH_DIR = RUNS_DIR / f"bench_{CONFIG}_{SHA7}"
    CFG = f"configs/spike_{CONFIG}.yaml"
    batch_cmd = [PY, "-m", "shipdoc", "predict", "batch", "--config", CFG,
                 "--bench-docs", BENCH_DOCS, "--bench-dir", str(BENCH_DIR),
                 "--decision", str(DECISION_PATH)]
    if DEV_RUN is not None:
        batch_cmd += ["--dev-run-dir", str(DEV_RUN)]
    if BATCH_SIZE is not None:
        batch_cmd += ["--batch-size", str(BATCH_SIZE)]
        print("=" * 78)
        print(f"MANUAL BATCH SIZE {BATCH_SIZE}: the bench is SKIPPED; nothing verifies that")
        print("this batch size gives the same outputs as batch 1 (the bench would).")
        print("=" * 78)
    rc, tail = run_stream(batch_cmd)
    if rc != 0:
        raise RuntimeError(f"BATCH SIZE REFUSED (exit {rc}): {tail[-3:]}. Test run NOT started.")
    decision = json.loads(DECISION_PATH.read_text())
    BATCH = int(decision["batch_size"])
    print(f"BATCH SIZE FOR THE TEST RUN: {BATCH} ({decision['source']}); dev contract: "
          f"{decision['contract']}")

else:
    print("skipped (batch size): NEED_INFERENCE is false")


In [ ]:
if NEED_INFERENCE:
    # Test run over the documents of this tab, one subprocess, RESUMABLE per doc: the CLI
    # appends one line to trace.jsonl and rewrites predictions.json / progress.json after every
    # document, and a resume skips the doc_ids already traced (the batch size is stored in
    # manifest.json). It refuses unless the smoke gate passed for this code. A disconnect loses at
    # most the window in flight.
    import re
    import threading
    import time
    from datetime import UTC, datetime

    if not SMOKE_PASSED or "BATCH" not in globals():
        raise RuntimeError("Smoke gate or batch decision missing: the test run is NOT started.")
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    OOM_RE = re.compile(r"out of memory|OutOfMemoryError", re.IGNORECASE)
    SESSIONS_PATH = RUN_DIR / "sessions.json"  # wall-clock log, one entry per Run all
    sessions = json.loads(SESSIONS_PATH.read_text()) if SESSIONS_PATH.is_file() else []
    THIS_DOCS = plan["shard_n_docs"]


    def save_json(path, obj) -> None:
        tmp = path.with_suffix(".tmp")
        tmp.write_text(json.dumps(obj, indent=1))
        tmp.replace(path)  # atomic: a disconnect never leaves a half-written file


    def read_progress() -> dict:
        try:
            return json.loads((RUN_DIR / "progress.json").read_text())
        except (OSError, ValueError):
            return {}


    def watch_progress(stop: threading.Event, every: float = 60.0) -> None:
        last, t_start = -1, time.time()
        while not stop.wait(every):
            p = read_progress()
            if p.get("done") != last:
                last = p.get("done")
                print(f"[progress] {p.get('done')}/{p.get('total')} docs, "
                      f"{(time.time() - t_start) / 60:.0f} min this session", flush=True)


    done_before = read_progress().get("done", 0)
    print(f"=== {SHARD_RUN_ID}: {done_before}/{THIS_DOCS} docs already done; resuming the rest "
          f"(batch size {BATCH}, shard {SHARD}) ===")
    session = {"start": datetime.now(UTC).isoformat(), "end": None, "seconds": None,
               "docs_done_at_start": done_before, "exit_code": None}
    sessions.append(session)
    save_json(SESSIONS_PATH, sessions)
    run_cmd = [PY, "-m", "shipdoc", "predict", "run", "--config", f"configs/spike_{CONFIG}.yaml",
               "--run-id", RUN_ID, "--shard", SHARD, "--decision", str(DECISION_PATH),
               "--smoke-status", str(SMOKE_STATUS_PATH),
               "--expect-docs", str(EXPECTED_DOCS), "--expect-pages", str(EXPECTED_PAGES)]
    stop = threading.Event()
    threading.Thread(target=watch_progress, args=(stop,), daemon=True).start()
    t0 = time.time()
    try:
        rc, tail = run_stream(run_cmd)
    finally:
        stop.set()
        session.update(end=datetime.now(UTC).isoformat(), seconds=round(time.time() - t0, 1))
        save_json(SESSIONS_PATH, sessions)
    session["exit_code"] = rc
    save_json(SESSIONS_PATH, sessions)
    if rc != 0:
        oom = OOM_RE.search("\n".join(tail))
        raise RuntimeError(
            f"Test run stopped (exit {rc}{', CUDA out of memory' if oom else ''}) after "
            f"{read_progress().get('done')}/{THIS_DOCS} docs; last output: {tail[-3:]}. Everything "
            "done so far is on Drive: Run all again resumes. If it stops at the same point, "
            "report back."
        )
    print(f"--- test run finished (exit 0) in {(time.time() - t0) / 60:.1f} min this session ---")

else:
    print("skipped (test run): NEED_INFERENCE is false")


In [ ]:
if NEED_INFERENCE:
    # DETERMINISM: 5 seeded test documents of this tab are decoded AGAIN in a fresh
    # process (new model load, new CUDA context). The generate calls of the first pass that held
    # those documents are REPLAYED exactly (same pages, same order inside the call), because a
    # batched decode can depend on the companions of a page; the serialized per-document predictions
    # must be byte-identical to the first pass. Counts only are printed.
    # A difference raises: the submission is not assembled as submittable.
    if not SMOKE_PASSED or "BATCH" not in globals():
        raise RuntimeError("Smoke gate or batch decision missing: the determinism pass is NOT run.")
    det_cmd = [PY, "-m", "shipdoc", "predict", "determinism", "--config",
               f"configs/spike_{CONFIG}.yaml", "--run-id", RUN_ID, "--shard", SHARD,
               "--decision", str(DECISION_PATH)]
    t_det = time.time()
    rc, tail = run_stream(det_cmd)
    if rc != 0:
        raise RuntimeError(f"DETERMINISM CHECK FAILED (exit {rc}): {tail[-3:]}. Report back "
                           "(counts only).")
    sessions.append({"start": None, "end": None, "seconds": round(time.time() - t_det, 1),
                     "exit_code": 0, "stage": "determinism"})
    save_json(SESSIONS_PATH, sessions)

else:
    print("skipped (determinism pass): NEED_INFERENCE is false")


In [ ]:
# MODE = 'merge': combine the K shard folders into the unsharded run folder (CPU).
# `python -m shipdoc merge-shards` REFUSES (and writes nothing) unless every shard is complete and
# the shards agree on code SHA, config hash, model revision, seed, batch size and doc list, no
# document is twice, and the merged counts are 200 docs / 280 pages. The assemble step below then
# also needs every shard's determinism pass.
if MODE == "merge":
    merge_cmd = [PY, "-m", "shipdoc", "merge-shards", "--config", f"configs/spike_{CONFIG}.yaml",
                 "--docs", str(TEST_DOCS_PATH), "--split", SPLIT, "--run-id", RUN_ID,
                 "--shards", str(SHARD_K), "--expected-docs", str(EXPECTED_DOCS),
                 "--expected-pages", str(EXPECTED_PAGES)]
    rc, tail = run_stream(merge_cmd)
    if rc != 0:
        raise RuntimeError(f"MERGE REFUSED (exit {rc}): {tail[-3:]}. Nothing was merged; fix the "
                           "shard that is named above (rerun it with Run all) and merge again.")
else:
    print("MODE = run: nothing to merge in this tab.")


In [ ]:
# ASSEMBLE + VALIDATION (CPU). Post-processing v1: R1, R2, R3 ON (default), the OCR
# cache passed, the frozen meta/slot_shapes.json. Reuse path: `shipdoc.reuse assemble` runs
# TWICE in fresh processes (the second into a scratch folder; the outputs must be byte-
# identical). Full path: `predict assemble` (every blocking check of 04: schema, exactly the
# 200 ids, determinism, batch contract, code SHA clean and equal to PINNED_SHA, ...). Then
# `shipdoc.reuse finalize` adds
# the v1 checks: rule switches all True, no R2 skipped for no_ocr / ocr_incomplete, shapes sha256
# equals meta/slot_shapes.json, OCR cache complete, OCR re-check, and (reuse) assemble-twice.
# A failure moves the predictions to test_predictions.REJECTED.json.
if MODE == "run" and SHARD_K > 1:
    print(f"Shard {SHARD} done. NEXT: when all {SHARD_K} tabs are COMPLETE, set MODE = 'merge' and "
          f"SHARD = '0/{SHARD_K}' in any tab and Run all: it merges, validates and assembles.")
else:
    if REUSE_OK:
        MODE_NAME = "reuse"
        RERUN_DIR = META_DIR / "assemble_rerun"
        asm_cmd = [PY, "-m", "shipdoc.reuse", "assemble", "--v0-dir", str(V0_DIR),
                   "--batch-size", str(BATCH_SIZE), "--shard", SHARD,
                   "--out-dir", str(SUBMISSION_DIR), "--schema", str(SCHEMA_PATH),
                   "--ocr-cache", str(OCR_ROOT), "--expect-docs", str(EXPECTED_DOCS),
                   "--expect-pages", str(EXPECTED_PAGES), "--expect-code-sha", PINNED_SHA]
        if SAMPLE_PATH.is_file():
            asm_cmd += ["--sample-submission", str(SAMPLE_PATH)]
        if DEV_RUN is not None:
            asm_cmd += ["--dev-run-dir", str(DEV_RUN)]
        if REUSE_ACK_SPIKE_DIFF:
            asm_cmd.append("--ack-spike-diff")
        rc, tail = run_stream(asm_cmd)
        if rc == 3:
            raise RuntimeError(f"REUSE REFUSED at assembly: {tail[-3:]}. Run all again to see the "
                               "decision; the full inference path runs when reuse is refused.")
        twice = [*asm_cmd[:asm_cmd.index("--out-dir")], "--out-dir", str(RERUN_DIR),
                 *asm_cmd[asm_cmd.index("--out-dir") + 2:]]
        rc2, tail2 = run_stream(twice)  # a fresh process: the outputs must match byte for byte
        fin_extra = ["--rerun-dir", str(RERUN_DIR)]
    else:
        MODE_NAME = "full"
        asm_cmd = [PY, "-m", "shipdoc", "predict", "assemble", "--config",
                   f"configs/spike_{CONFIG}.yaml", "--run-id", RUN_ID,
                   "--out-dir", str(SUBMISSION_DIR),
                   "--schema", str(SCHEMA_PATH), "--expect-docs", str(EXPECTED_DOCS),
                   "--expect-pages", str(EXPECTED_PAGES), "--expect-code-sha", PINNED_SHA,
                   "--require-stack", "--smoke-status", str(SMOKE_STATUS_PATH),
                   "--ocr-cache", str(OCR_ROOT)]
        if SAMPLE_PATH.is_file():
            asm_cmd += ["--sample-submission", str(SAMPLE_PATH)]
        if DEV_RUN is not None:
            asm_cmd += ["--dev-run-dir", str(DEV_RUN)]
        rc, tail = run_stream(asm_cmd)
        fin_extra = []
    fin_cmd = [PY, "-m", "shipdoc.reuse", "finalize", "--out-dir", str(SUBMISSION_DIR),
               "--mode", MODE_NAME, "--recheck", str(OCR_RECHECK_PATH),
               "--expect-pages", str(EXPECTED_PAGES), *fin_extra]
    rc_fin, tail_fin = run_stream(fin_cmd)
    if rc_fin != 0:
        raise RuntimeError(
            f"VALIDATION FAILED (assemble exit {rc}, v1 checks exit {rc_fin}): the files in "
            f"{SUBMISSION_DIR} are NOT submittable (test_predictions.REJECTED.json). Read "
            "validation_report.json there (counts and key paths only) and report back; do not "
            "submit."
        )


In [ ]:
# Completion banner. Reads only files on Drive: also runnable alone after a resume.
import json

bar = "=" * 78
BS = chr(92)
LOCAL_DEST = BS.join(["$SHIPDOC_SUBMISSIONS_DIR", f"v1_{SHA7}"]) + BS
report_path = SUBMISSION_DIR / "validation_report.json"
manifest_path = SUBMISSION_DIR / "manifest.json"
report = json.loads(report_path.read_text()) if report_path.is_file() else None
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
print(bar)
if report is None:
    print(f"NOT ASSEMBLED: {report_path.name} missing. Run the cells above (MODE='merge' after "
          "all shards).")
    print(f"DONE {SHARD_RUN_ID} assembled=False")
else:
    ok = bool(report["ok"])
    print("TEST PREDICTIONS v1 " + ("VALIDATED: SUBMITTABLE" if ok else "REJECTED: DO NOT SUBMIT"))
    print(bar)
    print(f"mode            : {report.get('mode')}  (reuse = the v0 model outputs, full = decoded)")
    print(f"schema ok       : {report['schema_ok']}")
    print(f"docs found      : {report['docs_found']}")
    print(f"determinism ok  : {report['determinism_ok']}")
    for name, c in report["checks"].items():
        print(f"  {'PASS' if c['ok'] else 'FAIL'}  {name}: {c.get('detail', '')}")
    reuse = manifest.get("reuse")
    if reuse:
        dp = reuse["decision"]["decode_path"]
        print(f"REUSE of {reuse['v0_dir']} (v0 code {str(reuse['v0_code_sha'])[:12]}): "
              "code identity REPLACED by")
        print(f"  decode-path blob fingerprint: {dp['files']} files identical, spike.py "
              f"{dp['spike']['state']}; config hash {manifest.get('config', {}).get('hash')}")
        print("  VLM determinism rests on v0's own check (manifest.determinism_v0, sha256 of its "
              "source recorded)")
    print(f"code {str(manifest.get('code_sha'))[:12]}  model {manifest.get('model', {}).get('id')} "
          f"@ {str(manifest.get('model', {}).get('revision'))[:12]}  config "
          f"{manifest.get('config', {}).get('hash')}")
    print(f"batch size {manifest.get('batch_size')} ({manifest.get('batch_size_source')}); "
          f"contract: {manifest.get('batch_size_contract')}")
    ocr_t = (manifest.get("ocr") or {}).get("timing") or {}
    print(f"OCR: {ocr_t.get('pages')} pages, {ocr_t.get('seconds_total')} s total, "
          f"{ocr_t.get('seconds_per_page_mean')} s/page mean, p95 "
          f"{ocr_t.get('seconds_per_page_p95')}, device {ocr_t.get('device')}")
    post = manifest.get("post_rules") or {}
    print(f"rules R1/R2/R3: touched docs {post.get('touched_docs')}, skipped {post.get('skipped')}")
    tm = manifest.get("timings", {})
    print(f"wall-clock {tm.get('wall_clock_s')} s over {tm.get('sessions')} session(s); model time "
          f"{tm.get('model_time_s')} s; smoke {tm.get('smoke_s')} s (from_v0_run: "
          f"{tm.get('from_v0_run', False)})")
    print(f"Drive folder: {SUBMISSION_DIR}")
    print("GG: download these files from that Drive folder:")
    wanted = ["test_predictions.json" if ok else "test_predictions.REJECTED.json",
              "trace.jsonl", "manifest.json", "validation_report.json", "rules.jsonl",
              "ocr_timing.json"]
    for name in wanted:
        path = SUBMISSION_DIR / name
        size = f"{path.stat().st_size / 1e6:8.2f} MB" if path.is_file() else "MISSING"
        print(f"  {name:<34}{size}")
    print(f"into the LOCAL folder  {LOCAL_DEST}")
    print("(outside the repo; submissions/ is gitignored). Never paste document values into chat.")
    print(f"DONE {SHARD_RUN_ID} assembled=True submittable={ok}")
print(bar)
